# CoQA + CNN/Daily Mail + WeeBit → Llama-3.1 judge (self-contained Colab)

Runs corpora through **`meta-llama/Llama-3.1-8B-Instruct`** with the same rubric as Shrishti’s `step3f_llm_judge.py` and writes **clean_dataset** CSVs.

| Corpus | Source | Output |
|--------|--------|--------|
| **CoQA** | HF `stanfordnlp/coqa` | `clean_dataset/coqa_train.csv` |
| **CNN/Daily Mail** | HF `cnn_dailymail` | `clean_dataset/cnn_dailymail_train.csv` |
| **WeeBit** | Kaggle CSVs on Drive (`data_train/val/test.csv`) | `clean_dataset/weebit_{train,val,test}.csv` |

## Before you run

1. **Runtime → Change runtime type → GPU (A100 or L4)** → **Restart session**
2. [Llama 3.1 license](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) + HF token
3. Upload WeeBit **raw Kaggle** files to `WEE_BIT_KAGGLE_DIR` (`data_train.csv`, `data_val.csv`, `data_test.csv`) — **not** `clean_dataset/`

## Run order (avoids vLLM init errors)

1. Config → Install → HF login → Drive mount → helpers → prepare splits  
2. **Skip** the GPU check cell until **after** judging (it initializes CUDA before vLLM)  
3. Judge cell → clean_dataset cell  

## Outputs on Drive

`DRIVE_ROOT` — `splits/`, `llm_judge/`, `clean_dataset/`, `manifest.json`

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MAX_COQA_STORIES = 2000
MAX_CNN_ARTICLES = 2000
MIN_CHARS = 80
SEED = 42

# Raw WeeBit from Kaggle (NOT clean_dataset — that folder holds judged outputs)
# Upload to Drive: BeyondFK/trail/weebit/data_train.csv, data_val.csv, data_test.csv
WEE_BIT_KAGGLE_DIR = "/content/drive/MyDrive/BeyondFK/trail/weebit"
WEE_BIT_FILES = {
    "weebit_train": "data_train.csv",
    "weebit_val": "data_val.csv",
    "weebit_test": "data_test.csv",
}
MAX_WEE_BIT_ROWS = None  # e.g. 500 per split for a quick test; None = all rows

# Set True to run CoQA + CNN only (no WeeBit FileNotFoundError)
SKIP_WEE_BIT = False

JUDGE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
MAX_MODEL_LEN = 2048
MAX_TOKENS = 8
GPU_MEM_FRAC = 0.85
ENABLE_PREFIX_CACHING = False  # False is more reliable on Colab + vLLM

WORK_ROOT = "/content/trail_judge_work"
SPLITS_DIR = f"{WORK_ROOT}/splits"
JUDGE_DIR = f"{WORK_ROOT}/llm_judge"
CLEAN_DIR = f"{WORK_ROOT}/clean_dataset"
HF_CACHE_DIR = f"{WORK_ROOT}/hf_cache"
LOGS_DIR = f"{WORK_ROOT}/logs"

DRIVE_ROOT = "/content/drive/MyDrive/BeyondFK/trail/judge_coqa_cnn"

CORPORA = [
    {"split_name": "coqa_train", "source_dataset": "coqa", "subject": "coqa", "clean_filename": "coqa_train.csv"},
    {"split_name": "cnn_dailymail_train", "source_dataset": "cnn_dailymail", "subject": "news", "clean_filename": "cnn_dailymail_train.csv"},
]
if not SKIP_WEE_BIT:
    CORPORA.extend([
        {"split_name": "weebit_train", "source_dataset": "weebit", "subject": "reading", "clean_filename": "weebit_train.csv"},
        {"split_name": "weebit_val", "source_dataset": "weebit", "subject": "reading", "clean_filename": "weebit_val.csv"},
        {"split_name": "weebit_test", "source_dataset": "weebit", "subject": "reading", "clean_filename": "weebit_test.csv"},
    ])

In [ ]:
import os

# Must be set before vLLM/CUDA (Colab + multiprocessing)
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

!pip install -q "datasets>=2.18" pandas huggingface_hub
!pip install -q "transformers>=4.44" accelerate
!pip install -q vllm codecarbon

for d in (WORK_ROOT, SPLITS_DIR, JUDGE_DIR, CLEAN_DIR, HF_CACHE_DIR, LOGS_DIR):
    os.makedirs(d, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR
print("Install OK. WORK_ROOT =", WORK_ROOT)

In [ ]:
# OPTIONAL — skip before Step 2 (judge). Running this initializes CUDA and can break vLLM.
# Run after judging, or only when debugging GPU availability.
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU — use Runtime → Change runtime type → GPU (A100/L4)")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Tip: skip this cell before the judge cell on a fresh runtime.")

In [ ]:
from huggingface_hub import login

login()
print("HF login OK")

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
for sub in ("splits", "llm_judge", "clean_dataset", "hf_cache", "logs"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
print("Drive:", DRIVE_ROOT)
print("WeeBit input dir:", WEE_BIT_KAGGLE_DIR)
if not SKIP_WEE_BIT:
    for split_name, fname in WEE_BIT_FILES.items():
        p = os.path.join(WEE_BIT_KAGGLE_DIR, fname)
        status = "OK" if os.path.exists(p) else "MISSING"
        print(f"  [{status}] {fname} -> {p}")
    missing = [f for f in WEE_BIT_FILES.values() if not os.path.exists(os.path.join(WEE_BIT_KAGGLE_DIR, f))]
    if missing:
        raise FileNotFoundError(
            f"Upload {missing} to {WEE_BIT_KAGGLE_DIR} or set SKIP_WEE_BIT=True"
        )
else:
    print("WeeBit skipped (SKIP_WEE_BIT=True)")

In [ ]:
# ── Shared rubric (same as step3f_llm_judge.py) ────────────────────────────
import hashlib
import json
import re
import shutil
import time

import pandas as pd
from datasets import load_dataset

RUBRIC = (
    "Classify the following text by its target reader's US education level.\n"
    "Choose exactly one of:\n"
    "- elementary  (US grades 1-5, simple vocabulary, short sentences)\n"
    "- middle      (US grades 6-8)\n"
    "- high        (US grades 9-12, advanced vocabulary, complex ideas)\n\n"
    "Text:\n{text}\n\n"
    "Reply with one word only: elementary, middle, or high."
)

CORE_COLS = [
    "split", "orig_split", "orig_idx", "source_dataset", "subject", "raw_label",
    "full_text", "education_level_original", "education_level_judge", "judge_raw_response",
]

TEXT_CANDIDATES = ["full_text", "text", "passage", "article", "content", "story", "document"]
LABEL_CANDIDATES = ["label", "y", "target", "readability", "level", "class", "labels", "grade"]

WEE_BIT_INT_MAP = {0: "elementary", 1: "elementary", 2: "middle", 3: "middle", 4: "high"}
WEE_BIT_STR_MAP = {
    "0": "elementary", "1": "elementary", "2": "middle", "3": "middle", "4": "high",
    "wrlevel2": "elementary", "wrlevel3": "elementary", "wrlevel4": "middle",
    "bitks3": "middle", "bitks": "middle", "bitgcse": "high", "gcse": "high",
    "elementary": "elementary", "middle": "middle", "high": "high",
}


def parse_label(raw: str) -> str:
    s = str(raw).strip().lower()
    s = re.sub(r"^[^a-z]+", "", s)
    if s.startswith("elem"):
        return "elementary"
    if s.startswith("mid"):
        return "middle"
    if s.startswith("high"):
        return "high"
    return "elementary"


def _domain_for(source_dataset: str) -> str:
    return "news" if source_dataset == "cnn_dailymail" else "reading"


def _base_row(full_text, split_name, source_dataset, subject, orig_idx, license_id, education_level, raw_label):
    key = hashlib.sha256(full_text.encode("utf-8")).hexdigest()
    return {
        "full_text": full_text,
        "education_level": education_level,
        "source_dataset": source_dataset,
        "domain": _domain_for(source_dataset),
        "label_source": f"{source_dataset}_kaggle" if source_dataset == "weebit" else f"{source_dataset}_unlabeled",
        "subject": subject,
        "raw_label": str(raw_label),
        "split": split_name,
        "orig_split": split_name,
        "orig_idx": orig_idx,
        "split_group": f"{source_dataset}::{key[:16]}",
        "source_license": license_id,
    }


def _pick_column(df, candidates, kind="column"):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    raise ValueError(f"Could not find {kind}. Have: {list(df.columns)}. Tried: {candidates}")


def weebit_label_to_level(raw) -> str:
    if pd.isna(raw):
        return "middle"
    if isinstance(raw, (int, float)) and not isinstance(raw, bool):
        return WEE_BIT_INT_MAP.get(int(raw), "middle")
    s = str(raw).strip().lower().replace(" ", "").replace("-", "")
    if s in WEE_BIT_STR_MAP:
        return WEE_BIT_STR_MAP[s]
    if s.isdigit():
        return WEE_BIT_INT_MAP.get(int(s), "middle")
    for k, v in WEE_BIT_STR_MAP.items():
        if k in s:
            return v
    return "middle"

In [ ]:
# ── Step 1: Prepare splits (CoQA + CNN + WeeBit) ─────────────────────────

def prepare_coqa(split_name: str = "coqa_train") -> str:
    path = f"{SPLITS_DIR}/{split_name}.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"[coqa] reuse {path} ({len(pd.read_csv(path))} rows)")
        return path
    print("[coqa] downloading stanfordnlp/coqa ...")
    ds = load_dataset("stanfordnlp/coqa", split="train")
    rows, seen = [], set()
    for ex in ds:
        story = str(ex.get("story") or "").strip()
        if len(story) < MIN_CHARS:
            continue
        key = hashlib.sha256(story.encode()).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        rows.append(_base_row(story, split_name, "coqa", "coqa", len(rows), "stanfordnlp/coqa", "middle", ""))
        if len(rows) >= MAX_COQA_STORIES:
            break
    df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
    df["orig_idx"] = range(len(df))
    df.to_csv(path, index=False)
    print(f"[coqa] wrote {len(df)} -> {path}")
    return path


def prepare_cnn_dailymail(split_name: str = "cnn_dailymail_train") -> str:
    path = f"{SPLITS_DIR}/{split_name}.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"[cnn] reuse {path} ({len(pd.read_csv(path))} rows)")
        return path
    print("[cnn] downloading cnn_dailymail 3.0.0 (train) ...")
    ds = load_dataset("cnn_dailymail", "3.0.0", split="train")
    rows, seen = [], set()
    for ex in ds:
        article = str(ex.get("article") or "").strip()
        if len(article) < MIN_CHARS:
            continue
        key = hashlib.sha256(article.encode()).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        rows.append(_base_row(article, split_name, "cnn_dailymail", "news", len(rows), "cnn_dailymail/3.0.0", "middle", ""))
        if len(rows) >= MAX_CNN_ARTICLES:
            break
    df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
    df["orig_idx"] = range(len(df))
    df.to_csv(path, index=False)
    print(f"[cnn] wrote {len(df)} -> {path}")
    return path


def prepare_weebit_from_kaggle(split_name: str, kaggle_filename: str) -> str:
    path = f"{SPLITS_DIR}/{split_name}.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"[weebit] reuse {path} ({len(pd.read_csv(path))} rows)")
        return path
    src = os.path.join(WEE_BIT_KAGGLE_DIR, kaggle_filename)
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"WeeBit file not found: {src}\n"
            f"Upload {kaggle_filename} to WEE_BIT_KAGGLE_DIR={WEE_BIT_KAGGLE_DIR}"
        )
    df_in = pd.read_csv(src)
    text_col = _pick_column(df_in, TEXT_CANDIDATES, "text")
    label_col = _pick_column(df_in, LABEL_CANDIDATES, "label")
    print(f"[weebit] {split_name}: columns text={text_col!r} label={label_col!r} from {src}")
    rows = []
    for _, r in df_in.iterrows():
        text = str(r[text_col]).strip()
        if len(text) < MIN_CHARS:
            continue
        raw_lab = r[label_col]
        level = weebit_label_to_level(raw_lab)
        rows.append(_base_row(text, split_name, "weebit", "reading", len(rows), "kaggle/weebit", level, raw_lab))
        if MAX_WEE_BIT_ROWS is not None and len(rows) >= MAX_WEE_BIT_ROWS:
            break
    if not rows:
        raise RuntimeError(f"[weebit] no rows from {src}")
    df = pd.DataFrame(rows)
    print(f"[weebit] original level dist: {dict(df['education_level'].value_counts())}")
    df.to_csv(path, index=False)
    print(f"[weebit] wrote {len(df)} -> {path}")
    return path


split_paths = {"coqa_train": prepare_coqa(), "cnn_dailymail_train": prepare_cnn_dailymail()}
if not SKIP_WEE_BIT:
    for split_name, fname in WEE_BIT_FILES.items():
        split_paths[split_name] = prepare_weebit_from_kaggle(split_name, fname)
else:
    print("[weebit] SKIP_WEE_BIT=True — skipping WeeBit splits")

for sn, p in split_paths.items():
    shutil.copy2(p, f"{DRIVE_ROOT}/splits/{sn}.csv")
print(f"Prepared {len(split_paths)} splits -> Drive/splits/")

In [ ]:
# ── Step 2: Llama judge (one model load, all splits) ─────────────────────
from codecarbon import EmissionsTracker
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams


def judge_split(split_name: str, llm, tokenizer, budget: int) -> str:
    split_csv = f"{SPLITS_DIR}/{split_name}.csv"
    judge_csv = f"{JUDGE_DIR}/{split_name}_judge.csv"
    df = pd.read_csv(split_csv)
    n = len(df)
    if os.path.exists(judge_csv):
        done = pd.read_csv(judge_csv)
        if len(done) == n:
            print(f"[judge] {split_name}: already complete ({n} rows)")
            shutil.copy2(judge_csv, f"{DRIVE_ROOT}/llm_judge/{split_name}_judge.csv")
            return judge_csv
    def build_prompt(text):
        ids = tokenizer(str(text), add_special_tokens=False)["input_ids"]
        if len(ids) > budget:
            text = tokenizer.decode(ids[:budget], skip_special_tokens=True)
        msgs = [{"role": "user", "content": RUBRIC.format(text=text)}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    prompts = [build_prompt(t) for t in df["full_text"].astype(str)]
    print(f"[judge] {split_name}: judging {len(prompts)} texts ...")
    t0 = time.time()
    outputs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS), use_tqdm=True)
    raws = [o.outputs[0].text for o in outputs]
    labels = [parse_label(r) for r in raws]
    rec = pd.DataFrame({
        "orig_split": df["orig_split"] if "orig_split" in df.columns else split_name,
        "orig_idx": df["orig_idx"] if "orig_idx" in df.columns else df.index,
        "source_dataset": df["source_dataset"].astype(str),
        "education_level": df["education_level"].astype(str),
        "llm_judge_label": labels,
        "judge_raw_response": raws,
    })
    rec.to_csv(judge_csv, index=False)
    agree = (rec["education_level"] == rec["llm_judge_label"]).mean()
    print(f"[judge] {split_name}: done in {(time.time()-t0)/60:.1f} min | agree orig={agree:.3f} | {dict(rec['llm_judge_label'].value_counts())}")
    shutil.copy2(judge_csv, f"{DRIVE_ROOT}/llm_judge/{split_name}_judge.csv")
    return judge_csv


tracker = EmissionsTracker(project_name="trail_coqa_cnn_weebit_judge", output_dir=LOGS_DIR, log_level="warning")
tracker.start()
try:
    print(f"Loading {JUDGE_MODEL} ...")
    tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
    llm = LLM(
        model=JUDGE_MODEL,
        dtype="bfloat16",
        enable_prefix_caching=ENABLE_PREFIX_CACHING,
        gpu_memory_utilization=GPU_MEM_FRAC,
        max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True,
    )
    empty = tokenizer.apply_chat_template(
        [{"role": "user", "content": RUBRIC.format(text="")}], tokenize=False, add_generation_prompt=True)
    overhead = len(tokenizer(empty, add_special_tokens=False)["input_ids"])
    budget = MAX_MODEL_LEN - overhead - MAX_TOKENS - 8
    print(f"max_text_tokens={budget}")
    for split_name in split_paths:
        judge_split(split_name, llm, tokenizer, budget)
finally:
    tracker.stop()
print("Judge step complete.")

In [ ]:
# ── Step 3: Build clean_dataset CSVs (Shrishti format) ────────────────────
manifest = {"corpora": [], "judge_model": JUDGE_MODEL}

for cfg in CORPORA:
    split_name = cfg["split_name"]
    split_csv = f"{SPLITS_DIR}/{split_name}.csv"
    judge_csv = f"{JUDGE_DIR}/{split_name}_judge.csv"
    clean_csv = f"{CLEAN_DIR}/{cfg['clean_filename']}"
    sdf = pd.read_csv(split_csv)
    jdf = pd.read_csv(judge_csv)
    if len(sdf) != len(jdf):
        raise RuntimeError(f"{split_name}: row mismatch {len(sdf)} vs {len(jdf)}")
    out = pd.DataFrame()
    out["split"] = [split_name] * len(sdf)
    out["orig_split"] = sdf.get("orig_split", split_name)
    out["orig_idx"] = sdf.get("orig_idx", range(len(sdf)))
    out["source_dataset"] = cfg["source_dataset"]
    out["subject"] = cfg["subject"]
    out["raw_label"] = sdf["raw_label"] if "raw_label" in sdf.columns else ""
    out["full_text"] = sdf["full_text"].astype(str)
    out["education_level_original"] = sdf["education_level"].astype(str)
    out["education_level_judge"] = jdf["llm_judge_label"].astype(str)
    out["judge_raw_response"] = jdf["judge_raw_response"].astype(str)
    out = out[CORE_COLS]
    out.to_csv(clean_csv, index=False)
    drive_clean = f"{DRIVE_ROOT}/clean_dataset/{cfg['clean_filename']}"
    shutil.copy2(clean_csv, drive_clean)
    dist = dict(out["education_level_judge"].value_counts())
    orig_dist = dict(out["education_level_original"].value_counts())
    print(f"\n[{cfg['source_dataset']}/{split_name}] n={len(out)}")
    print(f"  original (mapped): {orig_dist}")
    print(f"  judge: {dist}")
    print(f"  -> {drive_clean}")
    manifest["corpora"].append({
        "source_dataset": cfg["source_dataset"],
        "split_name": split_name,
        "n_rows": int(len(out)),
        "original_distribution": orig_dist,
        "judge_distribution": dist,
        "clean_file": cfg["clean_filename"],
    })

for mp in (f"{DRIVE_ROOT}/manifest.json", f"{WORK_ROOT}/manifest.json"):
    with open(mp, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

print("\n" + "=" * 60)
print("DONE. clean_dataset on Drive:")
for cfg in CORPORA:
    print(f"  {DRIVE_ROOT}/clean_dataset/{cfg['clean_filename']}")
print(f"  {DRIVE_ROOT}/manifest.json")
print("\nTrain/eval with column: education_level_judge")